# 青云扑克牌YOLOv8n训练（Kaggle GPU）
训练完下载 best_float32.tflite 或 best_int8.tflite

In [ ]:
# Cell 1: 安装依赖
!pip install -q ultralytics==8.3.0 roboflow
from IPython import display
display.clear_output()
import ultralytics
ultralytics.checks()
!nvidia-smi

In [ ]:
# Cell 2: 下载Roboflow公开扑克牌数据集（52类24k+张）
from roboflow import Roboflow
rf = Roboflow(api_key="cMVB8IAiArdKq5T3ZbTM")  # 公开匿名key
project = rf.workspace("augmented-startups").project("playing-cards-ow27d")
dataset = project.version(4).download("yolov8")
print('数据集位置:', dataset.location)

In [ ]:
# Cell 3: 训练YOLOv8n（约12-15分钟）
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # 加载官方nano预训练权重
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='qingyun_cards',
    patience=10,
    verbose=True
)
print('训练完成! mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

In [ ]:
# Cell 4: 导出TFLite（float32 + int8）
best_pt = 'runs/detect/qingyun_cards/weights/best.pt'
# Float32版本（精度高，约6MB，手机推理快）
model.export(format='tflite', imgsz=640)
# Int8量化版本（更小约3MB，速度更快，精度略降）
model.export(format='tflite', imgsz=640, int8=True)
import os
for f in ['best_float32.tflite', 'best_int8.tflite', 'best.pt']:
    p = f'runs/detect/qingyun_cards/weights/{f}'
    if os.path.exists(p):
        print(f'{f}: {os.path.getsize(p)/1024/1024:.2f} MB')

In [ ]:
# Cell 5: 打包并生成下载链接（Kaggle输出区可见）
import shutil, os
os.makedirs('/kaggle/working/qingyun_model', exist_ok=True)
for f in ['best_float32.tflite', 'best_int8.tflite', 'best.pt']:
    src = f'runs/detect/qingyun_cards/weights/{f}'
    dst = f'/kaggle/working/qingyun_model/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'已复制: {f}')
shutil.make_archive('/kaggle/working/qingyun_model', 'zip', '/kaggle/working/qingyun_model')
print('打包完成: /kaggle/working/qingyun_model.zip')
print('右侧Output面板找到 qingyun_model.zip → 点击下载 → 发给手抓饼')